[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/096.%20The%20Resilient%20Network%20Design%20%28Robust%20Optimization%29/P96-Tier-8_executed.ipynb)

---


# 96. Value-Aligned Network Design
## Tier 8: Ethical Constraints and Stakeholder Objectives

This notebook builds a small facility-location example in which candidate networks are evaluated against service, cost, environmental, social, and stakeholder criteria. The implementation uses a candidate-generation and mutation heuristic: it creates an initial set of facility combinations, scores them, and repeatedly generates new candidates guided by the best-scoring solution found so far.

The method is a stochastic heuristic, not an exact optimizer and not a robust-optimization model. In particular, it does not represent uncertain future scenarios. The name “resilient” refers to service-reliability indicators included in the score; it does not imply a formal guarantee against disruption.

### Goals, inputs, and outputs

- **Goal:** Select facilities that achieve strong stakeholder utility while accounting for ethical constraints.
- **Inputs:** Facility locations, capacities, costs, environmental-impact coefficients, demand-zone locations and attributes, stakeholder priorities, and ethical constraint thresholds.
- **Outputs:** A selected facility set, performance and compliance measures, stakeholder utility scores, a transparency report, an audit record, and diagnostic plots.
- **Method:** Heuristic search over facility subsets, with a fixed number of initial candidates and mutation steps. Candidate quality is measured by average stakeholder utility minus constraint penalties.

### Modeling assumptions and scope

Distances are Euclidean distances in the coordinate units supplied in the data. Service is represented by a distance-decayed capacity proxy; it is not a vehicle-routing or demand-allocation model. Each demand zone's service score is capped at 1, so the same zone cannot receive more than full service in the score. Facility emissions are annualized using an assumed utilization rate of 0.8 and an environmental-impact coefficient interpreted as tonnes of CO2 per capacity unit per day. These emissions are model estimates, not measured emissions.

Stakeholder priorities are treated as weights for normalized scores. The supplied priorities do not cover every metric for every stakeholder; each stakeholder's score therefore uses only the criteria explicitly present in that stakeholder's priorities. Satisfaction thresholds are reporting benchmarks and are not hard constraints in the search. Equity and emissions are hard constraints in this example, while vulnerable-population protection is a soft constraint with a penalty.

### Search method and scoring model

A candidate solution is a subset \(S\) of facility identifiers. Let \(U_k(S)\) be the utility score for stakeholder \(k\), and let \(C_j(S)\) indicate whether ethical constraint \(j\) is satisfied. The search ranks candidates using

$$
F(S) = \frac{1}{K}\sum_{k=1}^{K} U_k(S) - \sum_{j=1}^{J} p_j(S),
$$

where \(K\) is the number of stakeholders and \(p_j(S)\) is zero when constraint \(j\) is satisfied. A violated soft constraint incurs its stated penalty; a violated hard constraint incurs ten times its stated penalty. Thus, this implementation discourages violations but does not mathematically exclude them. Compliance is reported separately so that a high score cannot be mistaken for proof that all hard constraints are satisfied.

The procedure first creates cost-focused, service-focused, balanced, and random candidates. At each iteration, it evaluates every candidate and retains the best solution found. Mutation then modifies candidates, sometimes moving them toward the best solution and sometimes removing a facility. The method is heuristic: it can revisit candidates and does not guarantee that the globally best subset will be found. With \(I\) iterations, \(N\) candidates, and \(M\) facilities, the main search evaluates approximately \(I N\) candidates; the cost of each evaluation depends on the number of facilities and demand zones.

The implementation below makes the scoring and search steps explicit so that the assumptions, penalty mechanism, and reported results can be inspected.

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime
from itertools import combinations
import random
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

plt.style.use("seaborn-v0_8")


@dataclass
class Facility:
    """A candidate facility and its modeled impacts."""

    id: int
    x: float
    y: float
    capacity: int
    fixed_cost: float
    operational_status: bool = True
    environmental_impact: float = 0.0
    social_impact: Dict[str, float] = field(default_factory=dict)
    ethical_certifications: List[str] = field(default_factory=list)


@dataclass
class DemandZone:
    """A demand location and its modeled social characteristics."""

    id: int
    x: float
    y: float
    base_demand: float
    population_served: int = 0
    income_level: str = "medium"
    vulnerability_factors: List[str] = field(default_factory=list)
    community_priority: float = 1.0


@dataclass
class Stakeholder:
    """A stakeholder's criteria, weights, and reporting threshold."""

    id: int
    name: str
    stakeholder_type: str
    priorities: Dict[str, float]
    influence_level: float
    satisfaction_threshold: float = 0.8


@dataclass
class EthicalConstraint:
    """A modeled ethical requirement and its search penalty."""

    id: str
    name: str
    constraint_type: str
    description: str
    metric_type: str
    threshold_value: float
    penalty_weight: float
    is_hard_constraint: bool = True


@dataclass
class ConstitutionalAI:
    """Principles, constraints, and stakeholder weights for this example."""

    id: int
    name: str
    core_principles: List[str]
    ethical_constraints: List[EthicalConstraint]
    stakeholder_utilities: Dict[int, Dict[str, float]]
    compliance_monitor: bool = True
    transparency_level: float = 1.0


@dataclass
class ValueAlignedResult:
    """The selected network and its reported evaluation."""

    selected_facilities: List[Facility]
    objective_performance: Dict[str, float]
    ethical_compliance: Dict[str, bool]
    stakeholder_satisfaction: Dict[str, float]
    overall_utility: float
    transparency_report: Dict
    ethical_audit_trail: List[Dict]


In [ ]:
def create_ethical_ai_problem() -> Tuple[List[Facility], List[DemandZone], List[Stakeholder]]:
    """Create the facility, demand, and stakeholder data used in the example."""

    facilities = [
        Facility(1, 10, 20, 1000, 200000, True, 0.05,
                 {"employment": 50, "community_development": 30, "local_procurement": 20},
                 ["fair_trade", "environmental_certified"]),
        Facility(2, 30, 15, 1200, 250000, True, 0.08,
                 {"employment": 75, "community_development": 40, "local_procurement": 25},
                 ["sustainability_certified"]),
        Facility(3, 25, 35, 1100, 230000, True, 0.04,
                 {"employment": 40, "community_development": 50, "local_procurement": 15},
                 ["local_business_priority"]),
        Facility(4, 45, 30, 1300, 280000, True, 0.12,
                 {"employment": 60, "community_development": 35, "local_procurement": 30},
                 ["environmental_certified"]),
        Facility(5, 15, 50, 900, 180000, True, 0.03,
                 {"employment": 35, "community_development": 45, "local_procurement": 40},
                 ["community_first"]),
        Facility(6, 40, 10, 1400, 300000, True, 0.10,
                 {"employment": 80, "community_development": 30, "local_procurement": 20},
                 ["fair_trade", "environmental_certified"]),
    ]

    demand_zones = [
        DemandZone(1, 15, 25, 200, 50000, "low",
                   ["elderly", "limited_transport", "low_income"]),
        DemandZone(2, 35, 20, 300, 75000, "medium",
                   ["peak_hour_congestion"]),
        DemandZone(3, 25, 40, 250, 35000, "high",
                   ["elderly", "disabled_access"]),
        DemandZone(4, 40, 35, 350, 60000, "high",
                   ["limited_transport"]),
    ]

    stakeholders = [
        Stakeholder(1, "Community Coalition", "community",
                    {"service_equity": 0.4, "local_jobs": 0.3,
                     "environmental_quality": 0.2, "cost_affordability": 0.1},
                    0.8, 0.85),
        Stakeholder(2, "Environmental Alliance", "environmental",
                    {"carbon_footprint": 0.5, "biodiversity": 0.3,
                     "sustainability": 0.2},
                    0.7, 0.9),
        Stakeholder(3, "Business Association", "economic",
                    {"cost_efficiency": 0.4, "service_reliability": 0.3,
                     "market_competitiveness": 0.3},
                    0.9, 0.75),
        Stakeholder(4, "Regulatory Agency", "regulatory",
                    {"compliance": 0.6, "safety": 0.3,
                     "public_interest": 0.1},
                    0.95, 0.95),
    ]

    return facilities, demand_zones, stakeholders


facilities, demand_zones, stakeholders = create_ethical_ai_problem()

print("Value-Aligned Network Design Problem")
print(f"- Candidate facilities: {len(facilities)}")
print(f"- Demand zones: {len(demand_zones)}")
print(f"- Stakeholder groups: {len(stakeholders)}")
print("- Search: candidate generation followed by iterative mutation")

## Build the evaluation and search procedure

The class below separates the main tasks: measuring service and impacts, checking constraints, calculating stakeholder utilities, generating candidates, mutating candidates, and reporting the selected design. Keeping these steps visible makes it possible to trace how a candidate's data become a score and how that score guides the search.

For demand zone \(d\) and selected facility set \(S\), let \(r_{fd}\) be the Euclidean distance from facility \(f\) to zone \(d\). The service proxy is

$$
L_d(S) = \min\left(1,\; \frac{\sum_{f\in S}\min(c_f,q_d)\exp(-r_{fd}/1000)}{q_d}\right),
$$

where \(c_f\) is facility capacity and \(q_d\) is the zone's base demand. The exponential term reduces the contribution of distant facilities. The cap at 1 keeps the service level within the interval from zero to one. In the code, `_calculate_service_level` computes \(L_d(S)\), and `_calculate_total_service_coverage` computes its demand-weighted average.

The annual emissions proxy for \(S\) is

$$
E(S) = \sum_{f\in S} c_f\,u\,e_f\,365,
$$

where \(u=0.8\) is the assumed utilization and \(e_f\) is the facility's environmental-impact coefficient. These equations define the example's scoring proxies; they do not replace a detailed allocation, emissions-accounting, or uncertainty model.

In [ ]:
class ValueAlignedNetworkDesign:
    """Evaluate facility subsets and search them with candidate mutation."""

    def __init__(self, facilities: List[Facility], demand_zones: List[DemandZone],
                 stakeholders: List[Stakeholder]):
        self.facilities = facilities
        self.demand_zones = demand_zones
        self.stakeholders = stakeholders
        self.constitutional_ai = self._create_constitutional_ai()
        self.compliance_history: List[Dict[str, bool]] = []
        self.audit_trail: List[Dict] = []

    def _create_constitutional_ai(self) -> ConstitutionalAI:
        """Define the principles, constraints, and stakeholder weights."""
        principles = [
            "Network designs shall consider human welfare alongside cost.",
            "Environmental sustainability is a modeled hard requirement.",
            "Vulnerable populations receive enhanced protection.",
            "Service equity is assessed across the represented income groups.",
            "Decision criteria and results should be transparent.",
            "Stakeholder priorities are represented explicitly.",
        ]
        constraints = [
            EthicalConstraint(
                "EQUITY_001", "Service Equity Constraint", "equity",
                "Low-income service must be within 10% of high-income service.",
                "percentage", 0.1, 1000, True,
            ),
            EthicalConstraint(
                "ENV_001", "Carbon Emission Limit", "environmental",
                "Annual modeled emissions must not exceed 150,000 tonnes of CO2.",
                "absolute", 150000, 500, True,
            ),
            EthicalConstraint(
                "SOCIAL_001", "Vulnerable Population Protection", "social",
                "Average service for represented vulnerable zones should reach 0.9.",
                "threshold", 0.9, 750, False,
            ),
        ]
        utilities = {
            stakeholder.id: stakeholder.priorities.copy()
            for stakeholder in self.stakeholders
        }
        return ConstitutionalAI(
            id=1,
            name="Resilient Network Constitution",
            core_principles=principles,
            ethical_constraints=constraints,
            stakeholder_utilities=utilities,
            compliance_monitor=True,
            transparency_level=1.0,
        )

    def evaluate_solution_ethics(self, selected_facilities: List[int]) -> Dict[str, bool]:
        """Return a compliance result for each modeled ethical constraint."""
        return {
            constraint.id: self._check_constraint_compliance(
                constraint, selected_facilities
            )
            for constraint in self.constitutional_ai.ethical_constraints
        }

    def _check_constraint_compliance(self, constraint: EthicalConstraint,
                                      selected_facilities: List[int]) -> bool:
        """Dispatch a constraint check according to its modeled category."""
        if constraint.constraint_type == "equity":
            return self._check_equity_compliance(constraint, selected_facilities)
        if constraint.constraint_type == "environmental":
            return self._check_environmental_compliance(
                constraint, selected_facilities
            )
        if constraint.constraint_type == "social":
            return self._check_social_compliance(constraint, selected_facilities)
        raise ValueError(f"Unsupported constraint type: {constraint.constraint_type}")

    def _check_equity_compliance(self, constraint: EthicalConstraint,
                                 selected_facilities: List[int]) -> bool:
        """Compare mean service in low- and high-income demand zones."""
        levels: Dict[str, List[float]] = {}
        for demand in self.demand_zones:
            levels.setdefault(demand.income_level, []).append(
                self._calculate_service_level(demand, selected_facilities)
            )
        if "low" not in levels or "high" not in levels:
            return True
        low_service = float(np.mean(levels["low"]))
        high_service = float(np.mean(levels["high"]))
        if high_service == 0:
            return low_service == 0
        difference = abs(low_service - high_service) / high_service
        return difference <= constraint.threshold_value

    def _total_annual_emissions(self, selected_facilities: List[int]) -> float:
        """Calculate the annualized emissions proxy for selected facilities."""
        utilization = 0.8
        return sum(
            facility.capacity * utilization * facility.environmental_impact * 365
            for facility in self.facilities
            if facility.id in selected_facilities
        )

    def _check_environmental_compliance(self, constraint: EthicalConstraint,
                                         selected_facilities: List[int]) -> bool:
        """Check the modeled annual emissions against its threshold."""
        return self._total_annual_emissions(selected_facilities) <= constraint.threshold_value

    def _check_social_compliance(self, constraint: EthicalConstraint,
                                 selected_facilities: List[int]) -> bool:
        """Check the modeled protection level for vulnerable demand zones."""
        if constraint.id == "SOCIAL_001":
            return self._check_vulnerable_population_protection(
                selected_facilities, constraint.threshold_value
            )
        raise ValueError(f"Unsupported social constraint: {constraint.id}")

    def _check_vulnerable_population_protection(
        self, selected_facilities: List[int], threshold: float = 0.9
    ) -> bool:
        """Test mean service for zones marked elderly or disabled-access."""
        vulnerable_levels = [
            self._calculate_service_level(demand, selected_facilities)
            for demand in self.demand_zones
            if {"elderly", "disabled_access"} & set(demand.vulnerability_factors)
        ]
        return not vulnerable_levels or float(np.mean(vulnerable_levels)) >= threshold

    def _calculate_service_level(self, demand: DemandZone,
                                 selected_facilities: List[int]) -> float:
        """Calculate a distance-decayed, demand-capped service proxy."""
        if demand.base_demand <= 0:
            return 0.0
        selected = [
            facility for facility in self.facilities
            if facility.id in selected_facilities and facility.operational_status
        ]
        potential_service = 0.0
        for facility in selected:
            distance = np.hypot(facility.x - demand.x, facility.y - demand.y)
            potential_service += min(facility.capacity, demand.base_demand) * np.exp(
                -distance / 1000
            )
        return min(1.0, potential_service / demand.base_demand)

    def _calculate_total_service_coverage(self,
                                          selected_facilities: List[int]) -> float:
        """Calculate service as a base-demand-weighted average across zones."""
        total_demand = sum(demand.base_demand for demand in self.demand_zones)
        if total_demand <= 0:
            return 0.0
        served_demand = sum(
            self._calculate_service_level(demand, selected_facilities)
            * demand.base_demand
            for demand in self.demand_zones
        )
        return served_demand / total_demand

    def _calculate_service_equity(self, selected_facilities: List[int]) -> float:
        """Score similarity of service levels across the represented zones."""
        service_levels = [
            self._calculate_service_level(demand, selected_facilities)
            for demand in self.demand_zones
        ]
        if len(service_levels) < 2:
            return 1.0
        # Service levels lie in [0, 1], so this scaled variance score is bounded.
        return float(np.clip(1.0 - 4.0 * np.var(service_levels), 0.0, 1.0))

    def _calculate_local_jobs(self, selected_facilities: List[int]) -> float:
        """Normalize selected-facility employment by the available total."""
        maximum = sum(
            facility.social_impact.get("employment", 0)
            for facility in self.facilities
        )
        selected = sum(
            facility.social_impact.get("employment", 0)
            for facility in self.facilities
            if facility.id in selected_facilities
        )
        return selected / maximum if maximum > 0 else 0.0

    def _calculate_environmental_score(self,
                                       selected_facilities: List[int]) -> float:
        """Score lower emissions more favorably on a normalized scale."""
        maximum = self._total_annual_emissions(
            [facility.id for facility in self.facilities]
        )
        selected = self._total_annual_emissions(selected_facilities)
        return float(np.clip(1.0 - selected / maximum, 0.0, 1.0)) if maximum else 1.0

    def _calculate_carbon_efficiency(self,
                                     selected_facilities: List[int]) -> float:
        """Use the normalized environmental score for the carbon criterion."""
        return self._calculate_environmental_score(selected_facilities)

    def _calculate_cost_efficiency(self,
                                   selected_facilities: List[int]) -> float:
        """Balance service and cost in a dimensionless score between zero and one."""
        service = sum(
            self._calculate_service_level(demand, selected_facilities)
            for demand in self.demand_zones
        )
        cost_millions = sum(
            facility.fixed_cost
            for facility in self.facilities
            if facility.id in selected_facilities
        ) / 1_000_000
        denominator = service + cost_millions
        return service / denominator if denominator > 0 else 0.0

    def _calculate_service_reliability(self,
                                       selected_facilities: List[int]) -> float:
        """Approximate reliability from facility redundancy and dispersion."""
        selected = [
            facility for facility in self.facilities
            if facility.id in selected_facilities
        ]
        if not selected:
            return 0.0
        if len(selected) == 1:
            return 0.5
        redundancy = len(selected) / len(self.facilities)
        center_x = float(np.mean([facility.x for facility in selected]))
        center_y = float(np.mean([facility.y for facility in selected]))
        maximum_distance = max(
            np.hypot(facility.x - center_x, facility.y - center_y)
            for facility in selected
        )
        geographic_factor = min(1.0, maximum_distance / 50)
        return float(np.clip(redundancy * (1.0 - 0.3 * geographic_factor), 0.0, 1.0))

    def _calculate_compliance_score(self,
                                    selected_facilities: List[int]) -> float:
        """Return the fraction of modeled constraints satisfied."""
        compliance = self.evaluate_solution_ethics(selected_facilities)
        return (
            sum(compliance.values()) / len(compliance)
            if compliance else 1.0
        )

    def _calculate_safety_score(self,
                                selected_facilities: List[int]) -> float:
        """Use the reliability proxy as the modeled safety score."""
        return self._calculate_service_reliability(selected_facilities)

    def _calculate_public_interest_score(
        self, selected_facilities: List[int]
    ) -> float:
        """Combine coverage and normalized social benefits in a bounded score."""
        coverage = self._calculate_total_service_coverage(selected_facilities)
        maximum_benefit = sum(
            sum(facility.social_impact.values()) for facility in self.facilities
        )
        selected_benefit = sum(
            sum(facility.social_impact.values())
            for facility in self.facilities
            if facility.id in selected_facilities
        )
        social_score = selected_benefit / maximum_benefit if maximum_benefit else 0.0
        return (coverage + social_score) / 2

    def calculate_stakeholder_utilities(
        self, selected_facilities: List[int]
    ) -> Dict[str, float]:
        """Compute each stakeholder's weighted score from its stated priorities."""
        metric_functions = {
            "service_equity": self._calculate_service_equity,
            "local_jobs": self._calculate_local_jobs,
            "environmental_quality": self._calculate_environmental_score,
            "carbon_footprint": self._calculate_carbon_efficiency,
            "cost_efficiency": self._calculate_cost_efficiency,
            "service_reliability": self._calculate_service_reliability,
            "compliance": self._calculate_compliance_score,
            "safety": self._calculate_safety_score,
            "public_interest": self._calculate_public_interest_score,
        }
        utilities = {}
        for stakeholder in self.stakeholders:
            supported = [
                (metric_functions[name], weight)
                for name, weight in stakeholder.priorities.items()
                if name in metric_functions
            ]
            total_weight = sum(weight for _, weight in supported)
            if total_weight <= 0:
                utilities[stakeholder.name] = 0.0
                continue
            utilities[stakeholder.name] = sum(
                weight * function(selected_facilities)
                for function, weight in supported
            ) / total_weight
        return utilities

    def _calculate_penalty(self, compliance: Dict[str, bool]) -> float:
        """Apply the stated penalty, multiplying hard-constraint penalties by ten."""
        constraints = {
            constraint.id: constraint
            for constraint in self.constitutional_ai.ethical_constraints
        }
        penalty = 0.0
        for constraint_id, compliant in compliance.items():
            if not compliant:
                constraint = constraints[constraint_id]
                multiplier = 10 if constraint.is_hard_constraint else 1
                penalty += multiplier * constraint.penalty_weight
        return penalty

    def optimize_value_aligned_network(
        self, max_iterations: int = 100
    ) -> ValueAlignedResult:
        """Search candidate facility sets using the notebook's mutation heuristic."""
        if max_iterations < 1:
            raise ValueError("max_iterations must be at least 1")
        print("=" * 60)
        print("VALUE-ALIGNED NETWORK DESIGN SEARCH")
        print("=" * 60)
        print(f"Maximum iterations: {max_iterations}")
        print(f"Ethical constraints: {len(self.constitutional_ai.ethical_constraints)}")
        print(f"Stakeholders: {len(self.stakeholders)}")

        candidates = self._generate_candidate_solutions(20)
        print(f"\nEvaluating {len(candidates)} initial candidate solutions...")
        best_solution: Optional[List[int]] = None
        best_utility = -float("inf")

        for iteration in range(max_iterations):
            for candidate in candidates:
                utilities = self.calculate_stakeholder_utilities(candidate)
                compliance = self.evaluate_solution_ethics(candidate)
                average_utility = float(np.mean(list(utilities.values()))) if utilities else 0.0
                overall_utility = average_utility - self._calculate_penalty(compliance)
                if overall_utility > best_utility:
                    best_utility = overall_utility
                    best_solution = candidate.copy()

            if best_solution is None:
                raise RuntimeError("Candidate generation produced no solutions")
            if iteration < max_iterations - 1:
                candidates = self._ethical_mutation(candidates, best_solution)
            if iteration % 20 == 0 or iteration == max_iterations - 1:
                print(f"Iteration {iteration + 1:3d}: Best score = {best_utility:.3f}")

        final_utilities = self.calculate_stakeholder_utilities(best_solution)
        final_compliance = self.evaluate_solution_ethics(best_solution)
        self.compliance_history.append(final_compliance.copy())
        performance = self._calculate_performance_metrics(best_solution)
        transparency_report = self._generate_transparency_report(best_solution)
        audit_entry = {
            "timestamp": datetime.now(),
            "iteration": max_iterations,
            "solution": best_solution.copy(),
            "utilities": final_utilities.copy(),
            "compliance": final_compliance.copy(),
            "performance": performance.copy(),
            "utility": best_utility,
        }
        self.audit_trail.append(audit_entry)

        return ValueAlignedResult(
            selected_facilities=[
                facility for facility in self.facilities
                if facility.id in best_solution
            ],
            objective_performance=performance,
            ethical_compliance=final_compliance,
            stakeholder_satisfaction=final_utilities,
            overall_utility=best_utility,
            transparency_report=transparency_report,
            ethical_audit_trail=self.audit_trail.copy(),
        )

    def _generate_candidate_solutions(self,
                                      num_candidates: int) -> List[List[int]]:
        """Create cost-focused, service-focused, balanced, and random candidates."""
        if num_candidates <= 0:
            return []
        count = min(3, len(self.facilities))
        if count == 0:
            return []
        candidates = []
        cost_sorted = sorted(self.facilities, key=lambda item: item.fixed_cost)
        cost_candidate = [facility.id for facility in cost_sorted[:count]]
        best_service = self._find_best_service_solution(count)
        balanced = self._find_balanced_solution(count)
        for index in range(num_candidates):
            if index < 5:
                candidates.append(cost_candidate.copy())
            elif index < 10:
                candidates.append(best_service.copy())
            elif index < 15:
                candidates.append(balanced.copy())
            else:
                candidates.append([
                    facility.id
                    for facility in random.sample(self.facilities, count)
                ])
        return candidates

    def _find_best_service_solution(self,
                                    num_facilities: int) -> List[int]:
        """Enumerate fixed-size subsets and retain the highest-coverage subset."""
        best_solution: List[int] = []
        best_coverage = -float("inf")
        for subset in combinations(self.facilities, num_facilities):
            candidate = [facility.id for facility in subset]
            coverage = self._calculate_total_service_coverage(candidate)
            if coverage > best_coverage:
                best_coverage = coverage
                best_solution = candidate
        return best_solution

    def _find_balanced_solution(self,
                                num_facilities: int) -> List[int]:
        """Enumerate fixed-size subsets using the stated service-cost balance."""
        best_solution: List[int] = []
        best_score = -float("inf")
        for subset in combinations(self.facilities, num_facilities):
            candidate = [facility.id for facility in subset]
            service = self._calculate_total_service_coverage(candidate)
            cost_millions = sum(
                facility.fixed_cost for facility in subset
            ) / 1_000_000
            score = 0.7 * service - 0.3 * cost_millions
            if score > best_score:
                best_score = score
                best_solution = candidate
        return best_solution

    def _ethical_mutation(self, candidates: List[List[int]],
                          best_solution: List[int]) -> List[List[int]]:
        """Mutate candidates toward the incumbent or remove a weak facility."""
        mutated_candidates = []
        for candidate in candidates:
            if random.random() < 0.7:
                mutated = self._perform_ethical_mutation(candidate, best_solution)
                mutated_candidates.append(mutated)
            else:
                mutated_candidates.append(candidate.copy())
        return mutated_candidates

    def _perform_ethical_mutation(self, candidate: List[int],
                                  best_solution: List[int]) -> List[int]:
        """Apply one mutation while keeping a unique, valid facility subset."""
        mutated = candidate.copy()
        if random.random() < 0.5:
            additions = [
                facility_id for facility_id in best_solution
                if facility_id not in mutated
            ]
            if additions:
                mutated.append(random.choice(additions))
                if len(mutated) > 4:
                    mutated.pop(0)
        elif len(mutated) > 2:
            facility_to_remove = self._find_worst_facility(mutated)
            if facility_to_remove is not None:
                mutated.remove(facility_to_remove)
        if len(mutated) < min(2, len(self.facilities)):
            return self._generate_ethical_candidate()
        return mutated

    def _find_worst_facility(self, candidate: List[int]) -> Optional[int]:
        """Identify the candidate facility with the lowest contribution proxy."""
        if not candidate:
            return None
        return min(candidate, key=self._calculate_facility_contribution)

    def _calculate_facility_contribution(self, facility_id: int) -> float:
        """Combine a facility's service potential and modeled impacts."""
        facility = next(
            (item for item in self.facilities if item.id == facility_id), None
        )
        if facility is None:
            return 0.0
        service_potential = 0.0
        for demand in self.demand_zones:
            distance = np.hypot(facility.x - demand.x, facility.y - demand.y)
            service_potential += min(facility.capacity, demand.base_demand) * np.exp(
                -distance / 1000
            )
        total_demand = sum(demand.base_demand for demand in self.demand_zones)
        service_score = service_potential / total_demand if total_demand else 0.0
        maximum_social = sum(
            sum(item.social_impact.values()) for item in self.facilities
        )
        social_score = (
            sum(facility.social_impact.values()) / maximum_social
            if maximum_social else 0.0
        )
        maximum_emissions = self._total_annual_emissions(
            [item.id for item in self.facilities]
        )
        emissions = facility.capacity * 0.8 * facility.environmental_impact * 365
        environmental_score = (
            1.0 - emissions / maximum_emissions if maximum_emissions else 1.0
        )
        return 0.5 * service_score + 0.3 * social_score + 0.2 * environmental_score

    def _generate_ethical_candidate(self) -> List[int]:
        """Generate a candidate from facilities with high contribution scores."""
        ranked = sorted(
            self.facilities,
            key=lambda facility: self._calculate_facility_contribution(facility.id),
            reverse=True,
        )
        return [facility.id for facility in ranked[:min(3, len(ranked))]]

    def _calculate_performance_metrics(
        self, selected_facilities: List[int]
    ) -> Dict[str, float]:
        """Calculate service, cost, environmental, social, and reliability metrics."""
        selected_cost = sum(
            facility.fixed_cost
            for facility in self.facilities
            if facility.id in selected_facilities
        )
        maximum_social = sum(
            sum(facility.social_impact.values()) for facility in self.facilities
        )
        selected_social = sum(
            sum(facility.social_impact.values())
            for facility in self.facilities
            if facility.id in selected_facilities
        )
        return {
            "service_coverage": self._calculate_total_service_coverage(
                selected_facilities
            ),
            "cost_efficiency": self._calculate_cost_efficiency(selected_facilities),
            "annual_emissions_proxy": self._total_annual_emissions(
                selected_facilities
            ),
            "environmental_efficiency": self._calculate_environmental_score(
                selected_facilities
            ),
            "social_impact": selected_social / maximum_social if maximum_social else 0.0,
            "reliability": self._calculate_service_reliability(selected_facilities),
            "total_fixed_cost": selected_cost,
        }

    def _generate_transparency_report(
        self, selected_facilities: List[int]
    ) -> Dict:
        """Record the criteria, outcomes, and modeled risks for the selected set."""
        compliance = self.evaluate_solution_ethics(selected_facilities)
        selected_objects = [
            facility for facility in self.facilities
            if facility.id in selected_facilities
        ]
        risks = []
        if len(selected_objects) < 3:
            risks.append(
                f"Low redundancy indicator: {len(selected_objects)} facilities selected."
            )
        if len(selected_objects) >= 2:
            center_x = float(np.mean([facility.x for facility in selected_objects]))
            center_y = float(np.mean([facility.y for facility in selected_objects]))
            maximum_distance = max(
                np.hypot(facility.x - center_x, facility.y - center_y)
                for facility in selected_objects
            )
            if maximum_distance < 25:
                risks.append("Geographic concentration indicator: selected sites are close together.")
        return {
            "timestamp": datetime.now(),
            "selected_facilities": selected_facilities.copy(),
            "ethical_compliance": compliance,
            "stakeholder_utilities": self.calculate_stakeholder_utilities(
                selected_facilities
            ),
            "decision_rationale": [
                "The search ranks candidates using mean stakeholder utility minus constraint penalties.",
                "Service, cost, environmental impact, social impact, and reliability are reported.",
                "The search heuristic does not guarantee a globally optimal solution.",
            ],
            "risk_assessment": risks,
            "alternative_considerations": [
                "Cost-focused candidate generation prioritizes lower fixed-cost facilities.",
                "Service-focused candidate generation prioritizes modeled service coverage.",
                "Balanced candidate generation trades off service coverage and fixed cost.",
            ],
        }

    def generate_stakeholder_satisfaction_report(
        self, result: ValueAlignedResult
    ) -> Dict:
        """Compare each utility score with its stakeholder's reporting threshold."""
        analysis = {}
        improvement_opportunities = []
        stakeholder_by_name = {
            stakeholder.name: stakeholder for stakeholder in self.stakeholders
        }
        for name, score in result.stakeholder_satisfaction.items():
            stakeholder = stakeholder_by_name[name]
            threshold = stakeholder.satisfaction_threshold
            if score >= threshold:
                status = "MEETS_REPORTING_THRESHOLD"
                gap = f"{score - threshold:.1%} above threshold"
            else:
                status = "BELOW_REPORTING_THRESHOLD"
                gap = f"{threshold - score:.1%} below threshold"
                if stakeholder.stakeholder_type == "community":
                    improvement_opportunities.append(
                        "Review service equity, local employment, and affordability priorities."
                    )
                elif stakeholder.stakeholder_type == "environmental":
                    improvement_opportunities.append(
                        "Review environmental impact and sustainability priorities."
                    )
                elif stakeholder.stakeholder_type == "economic":
                    improvement_opportunities.append(
                        "Review the cost and service-reliability trade-off."
                    )
            analysis[name] = {"status": status, "gap": gap}
        scores = list(result.stakeholder_satisfaction.values())
        return {
            "timestamp": datetime.now(),
            "satisfaction_scores": result.stakeholder_satisfaction,
            "satisfaction_analysis": analysis,
            "improvement_opportunities": improvement_opportunities,
            "overall_satisfaction": float(np.mean(scores)) if scores else 0.0,
        }

    def create_comprehensive_visualization(
        self, result: ValueAlignedResult
    ) -> plt.Figure:
        """Plot compliance, utility scores, metrics, and facility locations."""
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle("Value-Aligned Network Design Results", fontsize=16)

        ax = axes[0, 0]
        constraint_ids = list(result.ethical_compliance)
        statuses = [result.ethical_compliance[item] for item in constraint_ids]
        colors = ["seagreen" if status else "firebrick" for status in statuses]
        ax.bar(constraint_ids, [1] * len(constraint_ids), color=colors)
        ax.set_ylim(0, 1.2)
        ax.set_ylabel("Status (1 = satisfied, 0 = violated)")
        ax.set_title("Ethical Constraint Compliance")
        ax.tick_params(axis="x", rotation=20)

        ax = axes[0, 1]
        names = list(result.stakeholder_satisfaction)
        values = [result.stakeholder_satisfaction[name] for name in names]
        ax.bar(names, values, color="steelblue")
        for stakeholder in self.stakeholders:
            ax.axhline(
                stakeholder.satisfaction_threshold,
                linestyle="--",
                alpha=0.55,
                label=f"{stakeholder.name} threshold",
            )
        ax.set_ylabel("Normalized utility score")
        ax.set_title("Stakeholder Utility and Reporting Thresholds")
        ax.tick_params(axis="x", rotation=25)
        ax.legend(fontsize=7)

        ax = axes[1, 0]
        metric_names = list(result.objective_performance)
        metric_values = [result.objective_performance[name] for name in metric_names]
        ax.bar(metric_names, metric_values, color="darkcyan")
        ax.set_ylabel("Metric value (see metric name for units)")
        ax.set_title("Performance Metrics")
        ax.tick_params(axis="x", rotation=30)

        ax = axes[1, 1]
        selected_ids = {facility.id for facility in result.selected_facilities}
        for facility in self.facilities:
            selected = facility.id in selected_ids
            ax.scatter(
                facility.x,
                facility.y,
                color="seagreen" if selected else "lightgray",
                marker="o" if selected else "x",
                s=140,
                edgecolors="black" if selected else "none",
            )
            ax.annotate(f"F{facility.id}", (facility.x, facility.y), xytext=(4, 4),
                        textcoords="offset points")
        for demand in self.demand_zones:
            ax.scatter(demand.x, demand.y, color="royalblue", marker="s", s=90)
            ax.annotate(f"D{demand.id}", (demand.x, demand.y), xytext=(4, 4),
                        textcoords="offset points")
        ax.set_xlabel("X coordinate (model units)")
        ax.set_ylabel("Y coordinate (model units)")
        ax.set_title("Facility and Demand Locations")
        ax.grid(True, alpha=0.3)
        ax.legend(handles=[
            Patch(facecolor="seagreen", label="Selected facility"),
            Patch(facecolor="lightgray", label="Unselected facility"),
            Patch(facecolor="royalblue", label="Demand zone"),
        ])

        fig.tight_layout()
        plt.show()
        return fig

    def generate_ethical_audit_report(self) -> Dict:
        """Summarize recorded search evaluations and final audit entries."""
        if not self.compliance_history:
            compliance_trend = "no_recorded_evaluations"
        else:
            recent = self.compliance_history[-5:]
            compliance_trend = (
                "all_recorded_constraints_satisfied"
                if all(all(entry.values()) for entry in recent)
                else "one_or_more_recorded_violations"
            )
        satisfaction_trend = "no_recorded_evaluations"
        if self.audit_trail:
            recent_scores = [
                float(np.mean(list(entry["utilities"].values())))
                for entry in self.audit_trail[-5:]
                if entry.get("utilities")
            ]
            if recent_scores:
                satisfaction_trend = "recorded_scores_available"
        return {
            "audit_timestamp": datetime.now(),
            "framework_version": "Illustrative value-aligned search model",
            "ethical_constraints_count": len(
                self.constitutional_ai.ethical_constraints
            ),
            "stakeholder_count": len(self.stakeholders),
            "compliance_history": self.compliance_history.copy(),
            "audit_entries": len(self.audit_trail),
            "compliance_trend": compliance_trend,
            "satisfaction_trend": satisfaction_trend,
        }


## Run the search and inspect its result

This cell runs the candidate search for 50 iterations, reports the selected facilities and their modeled measures, and creates the satisfaction, transparency, audit, and visualization summaries. The stakeholder utility values are normalized scores produced by the stated proxy metrics; they are not survey-derived satisfaction percentages. A threshold comparison therefore indicates only whether a modeled score meets its reporting benchmark.

The search can be stochastic because random candidates and mutations are used. No random seed is specified, so separate executions may explore different candidate sequences. The selected solution is the best candidate encountered during that execution, not a proof of global optimality.

In [ ]:
def run_value_aligned_optimization():
    """Run the search, report its evaluation, and return the model and result."""
    ethical_ai = ValueAlignedNetworkDesign(
        facilities, demand_zones, stakeholders
    )
    result = ethical_ai.optimize_value_aligned_network(max_iterations=50)

    print("\n" + "=" * 60)
    print("SEARCH RESULT")
    print("=" * 60)
    selected_ids = {facility.id for facility in result.selected_facilities}
    for facility in result.selected_facilities:
        print(f"Facility {facility.id}: location=({facility.x}, {facility.y})")
        print(f"  Fixed cost: ${facility.fixed_cost:,.0f}")
        print(f"  Environmental coefficient: {facility.environmental_impact:.3f}")
        print(f"  Social-impact total: {sum(facility.social_impact.values()):.1f}")

    print("\nPerformance metrics:")
    for metric, value in result.objective_performance.items():
        print(f"  {metric}: {value:.4f}")

    print("\nStakeholder utility scores:")
    stakeholder_by_name = {item.name: item for item in stakeholders}
    for name, score in result.stakeholder_satisfaction.items():
        threshold = stakeholder_by_name[name].satisfaction_threshold
        status = "meets threshold" if score >= threshold else "below threshold"
        print(f"  {name}: {score:.3f} ({status}; threshold {threshold:.2f})")

    print(f"\nOverall search score: {result.overall_utility:.3f}")
    print("Ethical constraint checks:")
    constraints = {
        constraint.id: constraint
        for constraint in ethical_ai.constitutional_ai.ethical_constraints
    }
    for constraint_id, compliant in result.ethical_compliance.items():
        status = "satisfied" if compliant else "violated"
        print(f"  {constraints[constraint_id].name}: {status}")

    satisfaction_report = ethical_ai.generate_stakeholder_satisfaction_report(result)
    print("\nStakeholder score summary:")
    print(f"  Mean utility score: {satisfaction_report['overall_satisfaction']:.3f}")
    for name, analysis in satisfaction_report["satisfaction_analysis"].items():
        print(f"  {name}: {analysis['status']} ({analysis['gap']})")

    print("\nDecision rationale:")
    for item in result.transparency_report["decision_rationale"]:
        print(f"  - {item}")
    print("Modeled risk indicators:")
    risks = result.transparency_report["risk_assessment"]
    for item in risks or ["No listed risk indicators under the checks in this example."]:
        print(f"  - {item}")

    audit_report = ethical_ai.generate_ethical_audit_report()
    print("\nAudit summary:")
    print(f"  Constraints: {audit_report['ethical_constraints_count']}")
    print(f"  Stakeholders: {audit_report['stakeholder_count']}")
    print(f"  Audit entries: {audit_report['audit_entries']}")
    print(f"  Compliance summary: {audit_report['compliance_trend']}")
    print(f"  Utility summary: {audit_report['satisfaction_trend']}")

    ethical_ai.create_comprehensive_visualization(result)
    assert selected_ids == {facility.id for facility in result.selected_facilities}
    assert set(result.ethical_compliance) == set(constraints)
    return ethical_ai, result


ethical_ai, value_result = run_value_aligned_optimization()

## Interpretation and limitations

The selected facilities are the best-scoring candidate encountered by the heuristic during this run. The performance table reports distinct quantities: service and normalized efficiency scores are dimensionless, fixed cost is in the units of the supplied cost data, and `annual_emissions_proxy` is an annualized model estimate under the stated coefficient and utilization assumptions. Constraint checks are Boolean results for the model's three defined tests.

A violated hard constraint remains possible because the search uses a penalty rather than a feasibility-enforcing mechanism. Likewise, satisfying all three checks establishes compliance only with these specific proxies and thresholds; it does not establish broader ethical, legal, or environmental compliance. Stakeholder thresholds are diagnostic benchmarks, not constraints, and the priority weights are illustrative inputs rather than empirically elicited preferences.

The reliability score is a simple indicator based on selected-facility count and geographic dispersion. It does not simulate facility failures, quantify recovery time, or model disrupted transport links. The model also omits demand allocation between facilities, facility opening decisions beyond selection, operational capacity interactions across zones, and explicit uncertain scenarios. These boundaries should be considered when interpreting the output or applying the model to operational decisions.

## Role of this modeling tier

Value-aligned network design makes social and environmental considerations visible alongside service and cost. Its value lies in explicitly recording the criteria, thresholds, penalties, and stakeholder weights used by the example, then reporting how the selected candidate performs under those definitions.

### Strengths

- Makes selected equity, environmental, and social checks part of candidate evaluation.
- Represents multiple stakeholder priorities rather than using cost alone.
- Produces interpretable performance measures and a basic audit record.
- Keeps candidate generation, scoring, mutation, and reporting as inspectable steps.

### Limitations

- Heuristic search does not guarantee a globally optimal design.
- Penalty-based ranking does not guarantee satisfaction of hard constraints.
- The service, environmental, and reliability measures are simplified proxies.
- Results depend on chosen data, weights, thresholds, and stochastic search path.
- The model does not provide formal robust-optimization guarantees under uncertainty.

### Suitable uses

This example is appropriate for learning how value criteria can be represented and examined in a network-design search. Before operational use, the data, units, service allocation assumptions, uncertainty representation, stakeholder weights, constraint definitions, and validation process would need to be established for the specific decision context.